# Gold Dimensions

## Purpose

I use this notebook as one of the transformation sources for the Gold Lakeflow
pipeline.

I create the reusable dimensions used by the claims analytical model:

- `dim_patient`
- `dim_provider`

Both dimensions are built from the validated Claims Silver dataset.

### Source

`health_insurance.silver.claims`

### Pipeline targets

- `health_insurance.gold.dim_patient`
- `health_insurance.gold.dim_provider`

### Modeling decisions

I keep the Kaggle claims patient domain separate from the FHIR patient domain.

Although both sources contain a field called `patient_id`, I do not have a
verified crosswalk proving that the identifiers represent the same people.
Joining them would create an artificial relationship.

I therefore build `dim_patient` only from the Claims Silver dataset. FHIR
Patient, Condition, and Encounter data remain a separate clinical domain and
can feed their own Gold analytical product later.

### Dimension grains

`dim_patient`

One row per Claims `patient_id`, using the demographic attributes from the
patient's latest available claim.

`dim_provider`

One row per distinct provider profile represented by:

- hospital ID
- provider type
- provider specialty
- provider city
- provider state

I generate deterministic surrogate keys for both dimensions so the later
`fact_claim` model can reference stable dimensional keys.

In [0]:
# importing the Lakeflow API and Spark functions used by the Gold dimensions.

from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "health_insurance"
CLAIMS_SOURCE = f"{CATALOG}.silver.claims"

## Patient dimension

building a current-state Patient dimension from the Claims domain.

A patient can appear on many claim rows, so I select the latest available
demographic snapshot using:

1. latest claim date
2. latest service date
3. highest claim ID as a deterministic tie-breaker

I also calculate the first and latest observed claim dates for analytical
context without changing the one-row-per-patient grain.

In [0]:
# defining the pipeline-managed Claims Patient dimension.

@dp.materialized_view(
    name="dim_patient",
    comment="Current claims-based patient dimension with one row per patient."
)
def dim_patient():

    claims_df = spark.read.table(
        CLAIMS_SOURCE
    )

    patient_window = (
        Window
        .partitionBy("patient_id")
        .orderBy(
            F.col("claim_date").desc_nulls_last(),
            F.col("service_date").desc_nulls_last(),
            F.col("claim_id").desc_nulls_last()
        )
    )

    patient_history_df = (
        claims_df
        .groupBy("patient_id")
        .agg(
            F.min("claim_date").alias("first_claim_date"),
            F.max("claim_date").alias("latest_claim_date"),
            F.count("*").alias("historical_claim_count")
        )
    )

    latest_patient_df = (
        claims_df

        .withColumn(
            "_patient_rank",
            F.row_number().over(patient_window)
        )

        .filter(
            F.col("_patient_rank") == 1
        )

        .select(
            "patient_id",
            "patient_age",
            "patient_age_group",
            "patient_gender",
            "patient_city",
            "patient_state"
        )
    )

    return (
        latest_patient_df

        .join(
            patient_history_df,
            on="patient_id",
            how="left"
        )

        .withColumn(
            "patient_key",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.lit("CLAIMS_PATIENT"),
                    F.col("patient_id").cast("string")
                ),
                256
            )
        )

        .select(
            "patient_key",
            "patient_id",
            "patient_age",
            "patient_age_group",
            "patient_gender",
            "patient_city",
            "patient_state",
            "first_claim_date",
            "latest_claim_date",
            "historical_claim_count"
        )

        .withColumn(
            "_gold_transformed_at",
            F.current_timestamp()
        )
    )

## Provider dimension

The Claims source does not contain a single provider identifier.

I therefore define the provider dimension grain as one distinct provider
profile composed of hospital ID, provider type, provider specialty, provider
city, and provider state.

I generate the same deterministic `provider_key` formula that I will reuse in
`fact_claim`.

This avoids assigning arbitrary generated IDs and ensures the dimension/fact
relationship can be reproduced on every pipeline refresh.

In [0]:
# defining the pipeline-managed provider-profile dimension.

@dp.materialized_view(
    name="dim_provider",
    comment="Distinct provider profiles derived from validated claims data."
)
def dim_provider():

    claims_df = spark.read.table(
        CLAIMS_SOURCE
    )

    provider_df = (
        claims_df

        .select(
            "hospital_id",
            "provider_type",
            "provider_specialty",
            "provider_city",
            "provider_state"
        )

        .dropDuplicates()
    )

    return (
        provider_df

        .withColumn(
            "provider_key",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.lit("CLAIMS_PROVIDER"),

                    F.coalesce(
                        F.col("hospital_id").cast("string"),
                        F.lit("UNKNOWN")
                    ),

                    F.coalesce(
                        F.col("provider_type"),
                        F.lit("UNKNOWN")
                    ),

                    F.coalesce(
                        F.col("provider_specialty"),
                        F.lit("UNKNOWN")
                    ),

                    F.coalesce(
                        F.col("provider_city"),
                        F.lit("UNKNOWN")
                    ),

                    F.coalesce(
                        F.col("provider_state"),
                        F.lit("UNKNOWN")
                    )
                ),
                256
            )
        )

        .select(
            "provider_key",
            "hospital_id",
            "provider_type",
            "provider_specialty",
            "provider_city",
            "provider_state"
        )

        .withColumn(
            "_gold_transformed_at",
            F.current_timestamp()
        )
    )

## Gold dimension outputs

When this notebook is added to the Gold Lakeflow pipeline, it defines:

### `dim_patient`

Grain: one row per Claims patient.

Key:

`patient_key`

Key formula:

`SHA2('CLAIMS_PATIENT' || patient_id)`

Primary descriptive attributes:

- patient age
- patient age group
- patient gender
- patient city
- patient state

Additional analytical context:

- first observed claim date
- latest observed claim date
- historical claim count

### `dim_provider`

Grain: one row per distinct provider profile.

Key:

`provider_key`

The key is derived deterministically from:

- hospital ID
- provider type
- provider specialty
- provider city
- provider state

I will reuse these exact key-generation rules in `fact_claim` so the fact table
references the dimensions consistently.

I do not manually persist or execute either dataset in this notebook. Lakeflow
will manage both Gold materialized views when the Gold pipeline is eventually
deployed and run.